# 04 - Feature Views: per-customer behavioral features

Declarative Feature Views (`databricks-feature-engineering` 0.18.x) over the silver Delta tables, keyed by `customer_id`. Features are registered in Unity Catalog, materialized to an offline Delta store for training, and to a Lakebase online store for low-latency lookup at serving.

**Why batch rather than streaming (option 3):** in this library, `StreamingMode()` only accepts features whose source is a Kafka or Kinesis-style stream (`StreamSource`/`KafkaSource`). It raises `ValueError` for Delta-table sources. Our silver tables are fed from files in a volume, so these features are **batch Feature Views on silver, refreshed on a short cron**: freshness is minutes, not seconds. The README lists what changes to move to `StreamingMode()`.

| Feature | Source | Function | Window |
| --- | --- | --- | --- |
| `failed_otp_count_10m` / `_1h` | silver_otp (status = failed) | Count | Sliding 10m / 5m slide; 1h / 5m slide |
| `failed_login_count_1h` | silver_login (success = false) | Count | Sliding 1h / 5m slide |
| `login_count_1h` | silver_login | Count | Sliding 1h / 5m slide |
| `distinct_devices_24h` | silver_login | ApproxCountDistinct(device_id) | Sliding 24h / 15m slide |
| `screen_view_count_10m` | silver_screen_view | Count | Sliding 10m / 5m slide |
| `avg_dwell_ms_10m` | silver_screen_view | Avg(dwell_ms) | Sliding 10m / 5m slide |
| `last_10_screens` | silver_screen_view | LastN(screen_name, 10) | Sliding 24h / 15m slide |
| `transfer_sum_7d` / `_30d` | silver_transfers | Sum(amount_usd) | Sliding 7d / 30d, 1h slide |
| `transfer_count_30d` | silver_transfers | Count | Sliding 30d / 1h slide |
| `corridor_transfer_count_30d` *(batch corridor history)* | silver_transfers, entity `customer_id + corridor` | Count | Sliding 30d / 1h slide |
| `distinct_corridors_30d` *(batch corridor history)* | silver_transfers | ApproxCountDistinct(corridor) | Sliding 30d / 1h slide |

Notes:
* **All windows are sliding.** For Delta-table sources, materialization rejects rolling-window aggregations other than `Last`. A sliding feature evaluated at time t uses the window that ends at the last slide boundary at or before t, so values lag by up to one slide. Batch materialization requires a slide of at least 5 minutes. Training uses the same semantics, which keeps training and serving consistent. Online freshness is additionally bounded by the refresh cron.
* **Consequence:** in-session account-takeover signals (an OTP burst seconds before the transfer) mostly fall after the last boundary and are not visible to these features. In-session signals need request-time features or true streaming (see README).
* **Freshness anchors.** Each online table holds one group of features (same source, filter, entity and slide). A group is rewritten only while one of its windows still has data, so a quiet customer's short-window values used to freeze at their last value. Each short-window group therefore has an `anchor_*` feature with a slightly longer window (2h or 25h). It keeps the group advancing until the short windows have emptied, and they are then written as NULL (treat as 0). Anchors are materialized but never used as model inputs.
* The window ends before the transfer time, so the transfer being scored never counts in its own transfer features.
* `device_id` is masked for non-admins with a deterministic token, so distinct-device counts are the same whether the materialization runs as an admin or not.

In [0]:
%pip install -q "databricks-feature-engineering>=0.17"
dbutils.library.restartPython()

In [0]:
import importlib.metadata as md
from datetime import timedelta
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import (
    Feature, DeltaTableSource, AggregationFunction, Count, Sum, Avg, ApproxCountDistinct, LastN,
    RollingWindow, SlidingWindow,
)

dbutils.widgets.text("catalog", "mlo_serverless_stable_catalog")
dbutils.widgets.text("schema", "fraud_rt")
dbutils.widgets.text("online_store_name", "fraud-rt-online")
dbutils.widgets.text("refresh_cron", "0 0/10 * * * ?")  # every 10 minutes (Quartz); runs take ~7 min, so 5 min skipped every other slot

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
ONLINE_STORE = dbutils.widgets.get("online_store_name")
REFRESH_CRON = dbutils.widgets.get("refresh_cron")

fe = FeatureEngineeringClient()
print("databricks-feature-engineering", md.version("databricks-feature-engineering"), "|", CATALOG, SCHEMA, ONLINE_STORE, REFRESH_CRON)

In [0]:
def src(table, filter_condition=None):
    return DeltaTableSource(catalog_name=CATALOG, schema_name=SCHEMA, table_name=table, filter_condition=filter_condition)

def agg(name, source, op, window, ts="event_ts", entity=("customer_id",), description=None):
    return Feature(source=source, function=AggregationFunction(op, window), entity=list(entity),
                   timeseries_column=ts, name=name, description=description)

m, h, d = (lambda x: timedelta(minutes=x)), (lambda x: timedelta(hours=x)), (lambda x: timedelta(days=x))

# All windows are SlidingWindow: for Delta-table sources, materialization rejects RollingWindow aggregations other
# than Last ("Only Last aggregation is supported for online-only RollingWindow feature"). A sliding feature evaluated
# at time t uses the window ending at the last slide boundary <= t, so values lag by up to one slide.
# Batch materialization (cron / table trigger) also requires a slide of at least 300s.
FEATURES = [
    # OTP / login behaviour (account-takeover signals)
    agg("failed_otp_count_10m", src("silver_otp", "status = 'failed'"), Count(input="event_id"), SlidingWindow(window_duration=m(10), slide_duration=m(5)),
        description="Failed OTP attempts in the last 10 minutes"),
    agg("failed_otp_count_1h", src("silver_otp", "status = 'failed'"), Count(input="event_id"), SlidingWindow(window_duration=h(1), slide_duration=m(5)),
        description="Failed OTP attempts in the last hour"),
    agg("failed_login_count_1h", src("silver_login", "success = false"), Count(input="event_id"), SlidingWindow(window_duration=h(1), slide_duration=m(5)),
        description="Failed logins in the last hour"),
    agg("login_count_1h", src("silver_login"), Count(input="event_id"), SlidingWindow(window_duration=h(1), slide_duration=m(5)),
        description="Login attempts in the last hour"),
    agg("distinct_devices_24h", src("silver_login"), ApproxCountDistinct(input="device_id"),
        SlidingWindow(window_duration=h(24), slide_duration=m(15)),
        description="Approx distinct devices used to log in over 24h (sliding, 15m slide)"),
    # Screen navigation (rapid screen hops)
    agg("screen_view_count_10m", src("silver_screen_view"), Count(input="event_id"), SlidingWindow(window_duration=m(10), slide_duration=m(5)),
        description="Screen views in the last 10 minutes"),
    agg("avg_dwell_ms_10m", src("silver_screen_view"), Avg(input="dwell_ms"), SlidingWindow(window_duration=m(10), slide_duration=m(5)),
        description="Average screen dwell time (ms) in the last 10 minutes"),
    agg("last_10_screens", src("silver_screen_view"), LastN(input="screen_name", n=10), SlidingWindow(window_duration=h(24), slide_duration=m(15)),
        description="Last 10 screens viewed in the last 24h, ordered by time"),
    # Transfer velocity
    agg("transfer_sum_7d", src("silver_transfers"), Sum(input="amount_usd"), SlidingWindow(window_duration=d(7), slide_duration=h(1)), ts="ts",
        description="Total USD sent in the last 7 days (excludes the transfer being scored)"),
    agg("transfer_sum_30d", src("silver_transfers"), Sum(input="amount_usd"), SlidingWindow(window_duration=d(30), slide_duration=h(1)), ts="ts",
        description="Total USD sent in the last 30 days (excludes the transfer being scored)"),
    agg("transfer_count_30d", src("silver_transfers"), Count(input="transfer_id"), SlidingWindow(window_duration=d(30), slide_duration=h(1)), ts="ts",
        description="Transfers in the last 30 days"),
    # Batch corridor history (30d)
    agg("corridor_transfer_count_30d", src("silver_transfers"), Count(input="transfer_id"), SlidingWindow(window_duration=d(30), slide_duration=h(1)),
        ts="ts", entity=("customer_id", "corridor"),
        description="Transfers by this customer to this corridor in the last 30 days (0 = new corridor)"),
    agg("distinct_corridors_30d", src("silver_transfers"), ApproxCountDistinct(input="corridor"),
        SlidingWindow(window_duration=d(30), slide_duration=h(1)), ts="ts",
        description="Approx distinct corridors used over 30d (sliding, 1h slide)"),
]
# Freshness anchors (materialization-only, never model inputs).
# Each online table is one group of features sharing source, filter, entity and slide. It is rewritten only while some
# window in the group still has data. Once a customer's group goes quiet, the table freezes at the last non-empty
# window, so short-window values stay stale (observed: failed_otp_count_1h served 9h after the last event). An anchor
# is a slightly longer window in the same group: it keeps the group advancing until the short windows have emptied,
# so they are written as NULL (= 0) instead of keeping their last value.
ANCHORS = [
    agg("anchor_failed_otp_2h", src("silver_otp", "status = 'failed'"), Count(input="event_id"),
        SlidingWindow(window_duration=h(2), slide_duration=m(5)), description="Freshness anchor for failed-OTP group (not a model input)"),
    agg("anchor_failed_login_2h", src("silver_login", "success = false"), Count(input="event_id"),
        SlidingWindow(window_duration=h(2), slide_duration=m(5)), description="Freshness anchor for failed-login group (not a model input)"),
    agg("anchor_login_2h", src("silver_login"), Count(input="event_id"),
        SlidingWindow(window_duration=h(2), slide_duration=m(5)), description="Freshness anchor for login group (not a model input)"),
    agg("anchor_screen_view_2h", src("silver_screen_view"), Count(input="event_id"),
        SlidingWindow(window_duration=h(2), slide_duration=m(5)), description="Freshness anchor for 10m screen group (not a model input)"),
    agg("anchor_login_25h", src("silver_login"), Count(input="event_id"),
        SlidingWindow(window_duration=h(25), slide_duration=m(15)), description="Freshness anchor for 24h device group (not a model input)"),
    agg("anchor_screen_view_25h", src("silver_screen_view"), Count(input="event_id"),
        SlidingWindow(window_duration=h(25), slide_duration=m(15)), description="Freshness anchor for 24h last-screens group (not a model input)"),
]
ALL_FEATURES = FEATURES + ANCHORS  # materialize all; train and serve on FEATURES only
print(len(FEATURES), "model features:", [f.name for f in FEATURES])
print(len(ANCHORS), "freshness anchors:", [f.name for f in ANCHORS])

In [0]:
# Labeled frame: one row per transfer. Features use two timeseries column names (event_ts for behavioral streams,
# ts for transfers), so the frame carries both, set to the transfer time.
transfers = spark.table(f"{CATALOG}.{SCHEMA}.silver_transfers")
labeled = (transfers.where("is_fraud = 1")
           .unionByName(transfers.where("is_fraud = 0").orderBy(F.rand(7)).limit(2000))
           .select("transfer_id", "customer_id", "corridor", "ts", F.col("ts").alias("event_ts"), "amount_usd", "is_fraud"))

training_set = fe.create_training_set(df=labeled, features=FEATURES, label="is_fraud",
                                      exclude_columns=["transfer_id", "ts", "event_ts", "corridor", "customer_id", "amount_usd"])
sample = fe.create_training_set(df=labeled, features=FEATURES, label="is_fraud").load_df()
sample_pd = sample.toPandas()
print(sample_pd.shape)
num = [f.name for f in FEATURES if f.name != "last_10_screens"]
display(spark.createDataFrame(sample_pd.groupby("is_fraud")[num].mean().round(2).reset_index()))

In [0]:
# Independent check: recompute three features with plain SQL point-in-time joins and compare row by row.
# SlidingWindow semantics: window [b - w, b) where b = transfer time floored to the slide (5m for 1h, 1h for 7d/30d).
spark.createDataFrame(sample_pd[["transfer_id", "customer_id", "corridor", "ts"]]).createOrReplaceTempView("fv_sample")
sql_vals = spark.sql(f"""
  SELECT s.transfer_id,
    (SELECT count(*) FROM {CATALOG}.{SCHEMA}.silver_otp o WHERE o.customer_id = s.customer_id AND o.status = 'failed'
       AND o.event_ts >= s.b5m - INTERVAL 1 HOUR AND o.event_ts < s.b5m) AS sql_failed_otp_count_1h,
    (SELECT coalesce(sum(t.amount_usd), 0) FROM {CATALOG}.{SCHEMA}.silver_transfers t WHERE t.customer_id = s.customer_id
       AND t.ts >= s.b1h - INTERVAL 7 DAYS AND t.ts < s.b1h) AS sql_transfer_sum_7d,
    (SELECT count(*) FROM {CATALOG}.{SCHEMA}.silver_transfers t WHERE t.customer_id = s.customer_id AND t.corridor = s.corridor
       AND t.ts >= s.b1h - INTERVAL 30 DAYS AND t.ts < s.b1h) AS sql_corridor_transfer_count_30d
  FROM (SELECT *, timestamp_seconds(floor(unix_timestamp(ts) / 300) * 300) AS b5m,
               timestamp_seconds(floor(unix_timestamp(ts) / 3600) * 3600) AS b1h FROM fv_sample) s""").toPandas()
cmp = sample_pd.merge(sql_vals, on="transfer_id")
for f in ["failed_otp_count_1h", "transfer_sum_7d", "corridor_transfer_count_30d"]:
    fv = cmp[f].fillna(0).astype(float)
    ref = cmp[f"sql_{f}"].astype(float)
    match = ((fv - ref).abs() < 0.01).mean()
    print(f"{f:30s} rows={len(cmp)}  exact-match rate={match:.4f}  mismatches={int((~((fv - ref).abs() < 0.01)).sum())}")

In [0]:
registered = []
for f in ALL_FEATURES:
    full = f"{CATALOG}.{SCHEMA}.{f.name}"
    try:
        registered.append(fe.get_feature(full_name=full))
        print("exists    ", full)
    except Exception:
        registered.append(fe.register_feature(feature=f, catalog_name=CATALOG, schema_name=SCHEMA))
        print("registered", full)

In [0]:
import time
store = fe.get_online_store(name=ONLINE_STORE)
if store is None:
    store = fe.create_online_store(name=ONLINE_STORE, capacity="CU_1")  # smallest size; synthetic prototype
    print("created online store", ONLINE_STORE)
for _ in range(60):  # wait up to ~20 min for the store to become available
    store = fe.get_online_store(name=ONLINE_STORE)
    state = str(getattr(store, "state", ""))
    if "AVAILABLE" in state:
        break
    time.sleep(20)
print(ONLINE_STORE, state, getattr(store, "capacity", None))

In [0]:
from databricks.feature_engineering.entities import OfflineStoreConfig, OnlineStoreConfig, CronSchedule

def already_materialized(full_name):
    return len(fe.list_materialized_features(feature_name=full_name)) > 0

to_materialize = [f for f in registered if not already_materialized(f"{CATALOG}.{SCHEMA}.{f.name}")]
print("to materialize:", [f.name for f in to_materialize])
if to_materialize:
    result = fe.materialize_features(
        features=to_materialize,
        offline_config=OfflineStoreConfig(catalog_name=CATALOG, schema_name=SCHEMA, table_name_prefix="fv_offline"),
        online_config=OnlineStoreConfig(catalog_name=CATALOG, schema_name=SCHEMA, table_name_prefix="fv_online",
                                        online_store_name=ONLINE_STORE),
        trigger=CronSchedule(quartz_cron_expression=REFRESH_CRON, timezone_id="UTC"),
        tags={"project": "fe_bar_fraud_rt"},
    )
    print(len(result or []), "materialized feature objects created")

In [0]:
rows = []
for f in ALL_FEATURES:
    full = f"{CATALOG}.{SCHEMA}.{f.name}"
    for mf in fe.list_materialized_features(feature_name=full):
        d = mf.as_dict() if hasattr(mf, "as_dict") else vars(mf)
        rows.append((f.name, "online" if d.get("is_online") or getattr(mf, "is_online", False) else "offline",
                     str(d.get("table_name") or d.get("offline_store_config") or d.get("online_store_config")),
                     str(d.get("pipeline_schedule_state") or d.get("state"))))
display(spark.createDataFrame(rows, "feature string, store string, target string, state string"))

In [0]:
# For each online feature group: the latest window written, how many model values are non-null, and how many
# customers truly had events in that window (SQL). With anchors, quiet groups advance until their short
# windows are empty, so stale values are cleared (NULL = 0) instead of being frozen at the last non-empty window.
from pyspark.sql import functions as F
online_tables = {}
for f in ALL_FEATURES:
    for mf in fe.list_materialized_features(feature_name=f"{CATALOG}.{SCHEMA}.{f.name}"):
        if mf.is_online:
            online_tables.setdefault(mf.as_dict()["table_name"], []).append(f.name)
rows = []
for tbl, feats in sorted(online_tables.items()):
    on = spark.table(tbl)
    offline = tbl.replace(".fv_online_", ".fv_offline_")
    ts_col = "ts" if "ts" in spark.table(offline).columns else "event_ts"
    boundary = spark.table(offline).agg(F.max(ts_col)).first()[0]
    for f in feats:
        rows.append((tbl.split(".")[-1], f, str(boundary), on.count(), on.where(F.col(f).isNotNull()).count()))
print("now:", spark.sql("SELECT current_timestamp()").first()[0])
display(spark.createDataFrame(rows, "online_table string, feature string, latest_window_end string, rows long, non_null long"))